# CallSense — Member 3 | 1. Semantic Retrieval
**هذه النوت بوك تقرأ ملف `all_samples.csv` مباشرة** وتبني بحثًا دلاليًا على نصوص التفريغ المتوفرة من الفريق.

**مهم:** الملف يحتوي `audio_id` و`transcript` فقط مع بيانات المجموعة والمسار. لا توجد توقيتات كلمات أو متحدثون؛ لذلك مصدر النتيجة سيكون `audio_id` وليس timestamp أو speaker.

شغّلي الخلايا بالترتيب. يفضّل اختيار GPU من:
`Runtime → Change runtime type → T4 GPU` (البحث يعمل على CPU أيضًا، لكن بناء embeddings أبطأ).

## 1) تثبيت المكتبات

In [1]:
!pip -q install sentence-transformers pandas numpy

## 2) رفع ملف الفريق

In [9]:
from google.colab import files
uploaded = files.upload()  # اختاري team_transcriptions_final
CSV_PATH = next(name for name in uploaded if name.lower().endswith(".csv"))
print("Loaded:", CSV_PATH)

Saving team_transcriptions_final (1).csv to team_transcriptions_final (1) (2).csv
Loaded: team_transcriptions_final (1) (2).csv


## 3) قراءة البيانات وفحصها

In [10]:
import pandas as pd

df = pd.read_csv(CSV_PATH)

# التأكد من وجود الأعمدة المطلوبة في الملف الجديد
required = {"call_id", "whisper_transcript"}
missing = required - set(df.columns)

if missing:
    raise ValueError(
        f"الأعمدة المطلوبة غير موجودة: {missing}. "
        f"الأعمدة الموجودة: {list(df.columns)}"
    )

# حذف الصفوف التي لا تحتوي على نص من Whisper
df = df.dropna(subset=["whisper_transcript"]).copy()

# تنظيف النصوص
df["whisper_transcript"] = (
    df["whisper_transcript"].astype(str).str.strip()
)

df = df[df["whisper_transcript"].ne("")].reset_index(drop=True)

# توحيد أسماء الأعمدة لتتوافق مع بقية أكواد المشروع
df["transcript"] = df["whisper_transcript"]
df["audio_id"] = df["call_id"].astype(str)

print("عدد النصوص:", len(df))
print("الأعمدة:", list(df.columns))

display(df.head(197))

عدد النصوص: 60
الأعمدة: ['call_id', 'audio_file', 'reference_transcript', 'whisper_transcript', 'dialect', 'transcript', 'audio_id']


,call_id,audio_file,reference_transcript,whisper_transcript,dialect,transcript,audio_id
0,1,call_001.m4a,Agent: السلام عليكم، معك خدمة العملاء، كيف اقد...,السلام عليكم معك خدمة العملك كيف أقدر ساعدك؟ و...,Code-switch,السلام عليكم معك خدمة العملك كيف أقدر ساعدك؟ و...,1
1,2,call_002.m4a,Agent: اهلا فيك بخمة العملاء تفضلي .\nCustomer...,أهلاً فيك في خدمة العملات؟ فضلي؟ أهلاً لو سمحت...,Saudi,أهلاً فيك في خدمة العملات؟ فضلي؟ أهلاً لو سمحت...,2
2,3,call_003.m4a,Agent: السلام عليكم، معاكم خدمة العملاء كيف اق...,السلام عليكم معك خدمة ماملكي فقد رأيك ساعدك و ...,Saudi,السلام عليكم معك خدمة ماملكي فقد رأيك ساعدك و ...,3
3,4,call_004.m4a,Agent: أهلًا وسهلًا، معك خدمة العملاء اتفضلي ....,أهلنا سهلاً معك لخدمة العملاء هتفضلي؟ مرحباً أ...,MSA,أهلنا سهلاً معك لخدمة العملاء هتفضلي؟ مرحباً أ...,4
4,5,call_005.m4a,Agent: هلا والله، معك خدمة العملاء، كيف أقدر أ...,هل وصلت لك مدينة كيف يمكنك اخذك؟ لقد سمحت الفا...,Saudi,هل وصلت لك مدينة كيف يمكنك اخذك؟ لقد سمحت الفا...,5
5,6,call_006.m4a,Agent: يا هلا، معك خدمة العملاء.\nCustomer: يا...,يا هلا معك خدمة العمانة يا أهلا مثالا قل لك أن...,Saudi,يا هلا معك خدمة العمانة يا أهلا مثالا قل لك أن...,6
6,7,call_007.m4a,\nAgent: السلام عليكم، كيف أقدر أخدمك؟\nCustom...,السلام عليكم كيف أقدر أخذيك وعليكم السلام الفت...,Saudi,السلام عليكم كيف أقدر أخذيك وعليكم السلام الفت...,7
7,8,call_008.m4a,Agent: hهلا، معك خدمة العملاء اتفضلي.\nCustome...,أهلاً معك خدمة العملاء أتفضلي؟ أهلاً عندي مشكل...,Code-switch,أهلاً معك خدمة العملاء أتفضلي؟ أهلاً عندي مشكل...,8
8,9,call_009.m4a,Agent: أهلًا، كيف أقدر أساعدك؟\nCustomer: السل...,أهلا كيف أقدر ساعدك؟ سلام عليكم لو سمحتي أنا ا...,Code-switch,أهلا كيف أقدر ساعدك؟ سلام عليكم لو سمحتي أنا ا...,9
9,10,call_010.m4a,Agent: السلام عليكم، معك الدعم الفني.\nCustome...,السلام عليكم وعكيد دعم الفني وعليكم السلام لو ...,Saudi,السلام عليكم وعكيد دعم الفني وعليكم السلام لو ...,10


## 4) تحميل نموذج embeddings متعدد اللغات

In [11]:
import torch, numpy as np, random
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
EMBED_MODEL = "intfloat/multilingual-e5-base"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

embedder = SentenceTransformer(EMBED_MODEL, device=device)
# E5 يتوقع prefix passage للنصوص و query للسؤال
passages = ["passage: " + t for t in df["transcript"].tolist()]
embeddings = embedder.encode(
    passages, batch_size=16, show_progress_bar=True,
    convert_to_numpy=True, normalize_embeddings=True
)
print("Embedding matrix:", embeddings.shape)

Device: cuda


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

... (7 outputs left out)

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Embedding matrix: (60, 768)


## 5) دالة البحث

In [12]:
def search_calls(query, top_k=5, group=None, dataset=None):
    q = embedder.encode(
        ["query: " + query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]
    scores = embeddings @ q
    results = df.copy()
    results["score"] = scores
    if group is not None and "group" in results.columns:
        results = results[results["group"].astype(str).str.lower() == str(group).lower()]
    if dataset is not None and "dataset" in results.columns:
        results = results[results["dataset"].astype(str).str.lower() == str(dataset).lower()]
    return results.sort_values("score", ascending=False).head(top_k)

question = "عميل يشتكي من مشكلة في الفاتورة"
hits = search_calls(question, top_k=5)
display(hits[[c for c in ["audio_id","dataset","group","score","transcript"] if c in hits.columns]])

,audio_id,score,transcript
2,3,0.851990,السلام عليكم معك خدمة ماملكي فقد رأيك ساعدك و ...
55,56,0.848471,أهلا بكم في خدمة العملاء كيف يمكنني مساعدتكم؟ ...
56,57,0.843694,أهلا بكم في خدمة العملاء كيف أخذمكم؟ مرحباً، ا...
23,24,0.840032,يا أهلا المثالب معك خدمت العملات كيف قد بخدمته...
0,1,0.839700,السلام عليكم معك خدمة العملك كيف أقدر ساعدك؟ و...


## 6) جرّبي أسئلتك

In [13]:
test_queries = [
    "مشكلة في الفاتورة أو مبلغ غير صحيح",
    "العميل يواجه مشكلة تقنية في الخدمة",
    "طلب إلغاء أو إيقاف الاشتراك",
    "customer complaint about billing",
]
for q in test_queries:
    print("\\nQUESTION:", q)
    display(search_calls(q, top_k=3)[[c for c in ["audio_id","group","score","transcript"] if c in df.columns or c=="score"]])

\nQUESTION: مشكلة في الفاتورة أو مبلغ غير صحيح


,audio_id,score,transcript
1,2,0.858233,أهلاً فيك في خدمة العملات؟ فضلي؟ أهلاً لو سمحت...
49,50,0.856182,السلام عليكم عن استفسار بخصوص فاتورة الأخيرة ل...
23,24,0.849873,يا أهلا المثالب معك خدمت العملات كيف قد بخدمته...


\nQUESTION: العميل يواجه مشكلة تقنية في الخدمة


,audio_id,score,transcript
56,57,0.853805,أهلا بكم في خدمة العملاء كيف أخذمكم؟ مرحباً، ا...
51,52,0.853325,أهلا بكم في خدمة العملاء كيف أختيمك؟ السلام عل...
7,8,0.850699,أهلاً معك خدمة العملاء أتفضلي؟ أهلاً عندي مشكل...


\nQUESTION: طلب إلغاء أو إيقاف الاشتراك


,audio_id,score,transcript
52,53,0.854010,مرحبا بكم، معكم خدمة العملاء، كيف أصدقائي مساع...
2,3,0.848424,السلام عليكم معك خدمة ماملكي فقد رأيك ساعدك و ...
57,58,0.848409,مرحبا بكم في الخدمة العملة كيف يمكنني مساعدتكم...


\nQUESTION: customer complaint about billing


,audio_id,score,transcript
2,3,0.811694,السلام عليكم معك خدمة ماملكي فقد رأيك ساعدك و ...
55,56,0.810398,أهلا بكم في خدمة العملاء كيف يمكنني مساعدتكم؟ ...
29,30,0.806509,السلام عليكم ومعكي خدمة العملاء كيف يقدروا ساع...


## 7) حفظ مخرجات الاسترجاع لاستخدامها في النوت بوك الثانية

In [14]:
import json, shutil, os
os.makedirs("retrieval_artifacts", exist_ok=True)
df.to_json("retrieval_artifacts/calls.json", orient="records", force_ascii=False, indent=2)
np.save("retrieval_artifacts/embeddings.npy", embeddings)
with open("retrieval_artifacts/config.json", "w", encoding="utf-8") as f:
    json.dump({"embedding_model": EMBED_MODEL, "rows": len(df)}, f, ensure_ascii=False, indent=2)
shutil.make_archive("retrieval_artifacts", "zip", "retrieval_artifacts")
files.download("retrieval_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## ملاحظات للتقرير
- نموذج الاسترجاع: `intfloat/multilingual-e5-base`.
- المقياس: cosine similarity على embeddings مطبّعة.
- مصدر الاستشهاد: `audio_id`.
- لا تكتبي أن النظام يستشهد بتوقيت أو متحدث؛ هذه الحقول غير موجودة في CSV.
- درجة التشابه ليست احتمالًا ولا تعني أن النتيجة صحيحة دائمًا؛ راجعي النتائج يدويًا.